In [ ]:
import math
from typing import Dict, List, Tuple
from .types import Block, EnergyConfig, MergeCandidate


def ceil_minutes(seconds: int) -> int:
    return int(math.ceil(seconds / 60.0))


def minutes_needed(energy_wh: float, charge_rate_wh_per_hour: float) -> int:
    if energy_wh <= 0:
        return 0

    wh_per_minute = charge_rate_wh_per_hour / 60.0
    return int(math.ceil(energy_wh / wh_per_minute))


def block_start_time(block: Block, trips: Dict[int, dict]) -> int:
    return int(trips[block[0]]["start"])


def block_end_time(block: Block, trips: Dict[int, dict]) -> int:
    return int(trips[block[-1]]["end"])


def arc_duration(arc_dict: dict, key, default: int = 0) -> int:
    arc = arc_dict.get(key)
    if arc is None:
        return default

    data = arc[-1]
    return int(data.get("duration", 0))


def arc_energy(arc_dict: dict, key, default: float = float("inf")) -> float:
    arc = arc_dict.get(key)
    if arc is None:
        return default

    data = arc[-1]
    return float(data.get("energy", 0.0))


def estimate_block_energy_need(
    block: Block,
    trips: Dict[int, dict],
    tt_best: Dict[Tuple[int, int], tuple],
    d5_best: Dict[int, tuple],
    td_best: Dict[int, tuple],
    config: EnergyConfig,
) -> float:
    """
    Energy required to leave the depot, serve the complete block,
    return to the depot, and still satisfy the minimum SoC reserve.
    """

    head = block[0]
    tail = block[-1]

    total = arc_energy(d5_best, head)
    total += float(trips[head].get("energy", 0.0))

    for u, v in zip(block, block[1:]):
        total += arc_energy(tt_best, (u, v))
        total += float(trips[v].get("energy", 0.0))

    total += arc_energy(td_best, tail)
    total += config.min_soc_reserve

    return total


def estimate_energy_after_block_return(
    block: Block,
    trips: Dict[int, dict],
    tt_best: Dict[Tuple[int, int], tuple],
    d5_best: Dict[int, tuple],
    d6_best: Dict[int, tuple],
    config: EnergyConfig,
) -> float:
    """
    Estimate the remaining battery energy after starting from the depot,
    serving this block, and returning to the depot.

    This is the arrival SoC at the depot after block i, used before deciding
    how much charging is needed before starting block j.
    """

    head = block[0]
    tail = block[-1]

    energy = config.battery_capacity

    energy -= arc_energy(d5_best, head)
    energy -= float(trips[head].get("energy", 0.0))

    for u, v in zip(block, block[1:]):
        energy -= arc_energy(tt_best, (u, v))
        energy -= float(trips[v].get("energy", 0.0))

    energy -= arc_energy(d6_best, tail)

    return energy


def build_block_merge_graph(
    blocks: List[Block],
    trips: Dict[int, dict],
    tt_best: Dict[Tuple[int, int], tuple],
    d5_best: Dict[int, tuple],
    d6_best: Dict[int, tuple],
    td_best: Dict[int, tuple],
    config: EnergyConfig,
) -> Tuple[Dict[int, List[int]], Dict[Tuple[int, int], MergeCandidate]]:
    """
    Build feasible block-to-block connections.

    A connection i -> j means:
        block i -> depot -> charge if needed -> depot -> block j

    This follows:
        1. block i must be able to return to the depot,
        2. block j must start from the same depot,
        3. the dwell time between both blocks must be sufficient,
        4. the required charging time must fit into the available dwell.
    """

    adjacency: Dict[int, List[int]] = {i: [] for i in range(len(blocks))}
    merge_info: Dict[Tuple[int, int], MergeCandidate] = {}

    for i, first_block in enumerate(blocks):
        tail_i = first_block[-1]

        if tail_i not in d6_best:
            continue

        depot_i, _, d6_data = d6_best[tail_i]
        return_to_depot_duration = int(d6_data.get("duration", 0))
        arrive_depot = block_end_time(first_block, trips) + return_to_depot_duration

        energy_after_return = estimate_energy_after_block_return(
            block=first_block,
            trips=trips,
            tt_best=tt_best,
            d5_best=d5_best,
            d6_best=d6_best,
            config=config,
        )

        # Block i is not valid for merging if it cannot return with reserve.
        if energy_after_return < config.min_soc_reserve:
            continue

        for j, second_block in enumerate(blocks):
            if i == j:
                continue

            head_j = second_block[0]

            if head_j not in d5_best:
                continue

            depot_j, _, d5_data = d5_best[head_j]

            # Blocks can only be merged through the same depot.
            if depot_i != depot_j:
                continue

            pull_out_duration = int(d5_data.get("duration", 0))
            latest_depart = block_start_time(second_block, trips) - pull_out_duration

            if config.allow_equal_time:
                if arrive_depot > latest_depart:
                    continue
            else:
                if arrive_depot >= latest_depart:
                    continue

            available_seconds = latest_depart - arrive_depot
            available_minutes = max(0, available_seconds // 60)

            energy_needed_for_second_block = estimate_block_energy_need(
                block=second_block,
                trips=trips,
                tt_best=tt_best,
                d5_best=d5_best,
                td_best=td_best,
                config=config,
            )

            # If block j cannot be operated even from a full battery, skip it.
            if energy_needed_for_second_block > config.battery_capacity:
                continue

            # Charge only the missing energy, not the whole energy requirement
            # of the second block.
            required_charge_energy = max(
                0.0,
                energy_needed_for_second_block - energy_after_return,
            )

            # Do not charge above battery capacity.
            max_charge_possible = max(
                0.0,
                config.battery_capacity - energy_after_return,
            )

            if required_charge_energy > max_charge_possible:
                continue

            required_charge_minutes = minutes_needed(
                energy_wh=required_charge_energy,
                charge_rate_wh_per_hour=config.charge_rate_wh_per_hour,
            )

            if required_charge_minutes <= available_minutes:
                adjacency[i].append(j)
                merge_info[(i, j)] = MergeCandidate(
                    from_block=i,
                    to_block=j,
                    arrive_depot_time=arrive_depot,
                    latest_depart_time=latest_depart,
                    required_charge_minutes=required_charge_minutes,
                    depot_id=int(depot_i),
                )

    return adjacency, merge_info